<a href="https://colab.research.google.com/github/camelia409/Redrob/blob/main/app/demo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redrob Candidate Ranker — Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camelia409/Redrob/blob/main/app/demo_colab.ipynb)

A simplified ranking demo that runs on 50 sample candidates. Uses BM25 keyword retrieval, 30 engineered features across 7 families, a weighted rerank, and template-based grounded reasoning. No LLM is called at any point. Runs on a free Colab CPU in about 30 seconds after dependencies install.

## How to run

Choose Runtime, then Run all. Wait about 30 seconds. The last cell prints the top 10 candidates with their scores and reasons.

In [1]:
# Cell 1 — Install deps + clone repo
!pip install -q rank-bm25==0.2.2 orjson==3.10.15 pyyaml==6.0.2 pandas==2.2.3 numpy==2.1.3
!git clone https://github.com/camelia409/Redrob.git /content/redrob
%cd /content/redrob
import sys
sys.path.insert(0, "/content/redrob")
print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 77.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.
langgraph-sdk 0.4.2 requires orjson>=3.11.5, but you have orjson 3.10.15 which is inco

In [3]:
!pip install -q python-docx==1.1.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 13.0 MB/s eta 0:00:00


In [4]:
# Cell 2 — Load JD + bundled sample candidates
from src.ingestion.jd import get_jd_text
from src.ingestion.loader import load_candidates_sample

jd = get_jd_text()
print("=" * 60)
print("JOB DESCRIPTION (first 600 chars):")
print("=" * 60)
print(jd[:600] + "...\n")

candidates = load_candidates_sample(50)
print(f"Loaded {len(candidates)} sample candidates.")
print(f"First candidate: {candidates[0]['candidate_id']} - {candidates[0]['profile'].get('current_title', '?')}")

JOB DESCRIPTION (first 600 chars):
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)
Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the com...

Loaded 50 sample candidates.
First candidate: CAND_0000001 - Backend Engineer


In [7]:
# Cell 3 — Rank candidates (self-contained weight loader)
from src.ranking.baselines import BM25Ranker
from src.features.extractor import FeatureExtractor
from src.ranking.weighted_reranker import WeightedReranker
from src.validation.honeypots import honeypot_score_gates_only, run_all_checks
from src.reasoning.generator import generate
from pathlib import Path
import yaml
import pandas as pd

# Load weights directly from YAML
with open("/content/redrob/configs/reranker_weights_v1.yaml") as f:
    weights_cfg = yaml.safe_load(f)
weights = weights_cfg.get("weights_v1", weights_cfg)  # handles either shape

# BM25
bm25 = BM25Ranker(jd_text=get_jd_text())
bm25_out = bm25.rank(candidates)
bm25_map = dict(bm25_out)

# Feature extraction (BM25 proxy for dense in demo)
hp_scores = {c["candidate_id"]: honeypot_score_gates_only(c) for c in candidates}
fe = FeatureExtractor()
df = fe.extract_batch(candidates, bm25_map, bm25_map, hp_scores)

# Weighted rerank
reranker = WeightedReranker(weights)
df["reranker_score"] = reranker.score(df)

# Add honeypot_score column so the gate step can read it
df["honeypot_score"] = df["candidate_id"].map(hp_scores)

df.loc[df["honeypot_score"] >= 3, "reranker_score"] -= 1e6
df = df.sort_values(by=["reranker_score", "candidate_id"], ascending=[False, True]).reset_index(drop=True)

top = df.head(10).copy()
top["rank"] = range(1, 11)
print(f"✅ Ranked {len(candidates)} candidates.\n")

✅ Ranked 50 candidates.



In [8]:
# Cell 4 — Show top-10 with reasonings
for i, row in top.iterrows():
    cid = row["candidate_id"]
    c = next(x for x in candidates if x["candidate_id"] == cid)
    p = c.get("profile", {})
    skills = ", ".join(s.get("name", "?") for s in c.get("skills", [])[:3])

    hp_flags = [k for k, (t, _) in run_all_checks(c).items() if t]
    features_row = row.to_dict()
    reason = generate(c, rank=i+1, score=float(row["reranker_score"]),
                      features_row=features_row, honeypot_flags=hp_flags)

    print(f"#{i+1}  {cid}  |  score={row['reranker_score']:.3f}")
    print(f"    {p.get('current_title', '?')} at {p.get('current_company', '?')} — {p.get('years_of_experience', 0):.1f} YoE — {p.get('location', '?')}")
    print(f"    Skills: {skills}")
    print(f"    💬 {reason}")
    print()

# Save as CSV
top[["candidate_id", "rank", "reranker_score"]].to_csv("/content/demo_ranking.csv", index=False)
print("Saved: /content/demo_ranking.csv")

#1  CAND_0000031  |  score=0.995
    Recommendation Systems Engineer at Swiggy — 6.0 YoE — Hyderabad, Telangana
    Skills: Go, MLflow, FAISS
    💬 Standout Recommendation Systems Engineer at Swiggy; 6.0 years of relevant experience; solid command of Pinecone and Information Retrieval

#2  CAND_0000025  |  score=0.909
    Frontend Engineer at Tech Mahindra — 7.3 YoE — Vizag, Andhra Pradesh
    Skills: JavaScript, Spark, GCP
    💬 Standout Frontend Engineer at Tech Mahindra; 7.3 years of relevant experience; solid command of LangChain and Data Pipelines

#3  CAND_0000001  |  score=0.907
    Backend Engineer at Mindtree — 6.9 YoE — Toronto
    Skills: Tailwind, NLP, Image Classification
    💬 Senior Backend Engineer with 6.9 years and clear ML production evidence; relevant expertise: TTS, Image Classification, Fine-tuning LLMs

#4  CAND_0000014  |  score=0.891
    Frontend Engineer at Zomato — 8.4 YoE — Hyderabad, Telangana
    Skills: FAISS, BigQuery, React
    💬 Standout Frontend Engin

In [ ]:
# Cell 5 — Upload your own candidates.jsonl (≤100 candidates)
from google.colab import files
import orjson

uploaded = files.upload()  # opens file picker
if uploaded:
    fname = list(uploaded.keys())[0]
    with open(fname, "rb") as f:
        custom_candidates = [orjson.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(custom_candidates)} candidates from upload.")
    # Re-run cells 3 + 4 using `custom_candidates` instead of `candidates`